# Why are Japan's regions losing people?

Data: Social and Demographic Statistics System (SSDS), Statistics Bureau of Japan, via the e-Stat API.
47 prefectures, 1975–2024.

1. Get and prepare the data
2. Feature selection: which statistics matter most?
3. ANOVA: does population change differ between regions and between periods?
4. Prediction: population 5 and 10 years ahead

## 1. Get and prepare the data

In [ ]:
# Setup: libraries and the e-Stat key (stored as a Colab secret named ESTAT_APP_ID)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from urllib.parse import quote
from google.colab import userdata

APP_ID = userdata.get("ESTAT_APP_ID")

In [ ]:
# Function: download one statistic from e-Stat as a table (area, year, value)
def get_estat(table_id, item_code):
    url = ("https://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData"
           f"?appId={APP_ID}&lang=E&statsDataId={table_id}&cdCat01={quote(item_code)}"
           "&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=N&annotationGetFlg=N"
           "&sectionHeaderFlg=2&replaceSpChars=0")
    df = pd.read_csv(url, dtype={"area_code": str})
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.rename(columns={"AREA": "area", "SURVEY YEAR": "year"})
    return df[["area_code", "area", "year", "value"]]

# test
get_estat("0000010101", "A1101").head()

In [ ]:
# The statistics we want: one or two per topic (table_id + item_code from e-Stat)
tables = {
    # Population
    "pop":               {"table_id": "0000010101", "item_code": "A1101"},     # total population
    "births":            {"table_id": "0000010101", "item_code": "A4101"},     # births
    "deaths":            {"table_id": "0000010101", "item_code": "A4200"},     # deaths
    "moves_in":          {"table_id": "0000010101", "item_code": "A5101"},     # people moving in
    "moves_out":         {"table_id": "0000010101", "item_code": "A5102"},     # people moving out
    "pop_65plus":        {"table_id": "0000010101", "item_code": "A1303"},     # people aged 65+
    # Family
    "fertility":         {"table_id": "0000010201", "item_code": "#A05203"},   # babies per woman
    "marriage_rate":     {"table_id": "0000010201", "item_code": "#A06601"},   # marriages per 1,000 people
    "single_households": {"table_id": "0000010201", "item_code": "#A06205"},   # people living alone (%)
    # Land and weather
    "density":           {"table_id": "0000010201", "item_code": "#A01202"},   # people per km2 of habitable land
    "snow_days":         {"table_id": "0000010202", "item_code": "#B02304"},   # days of snow per year
    "temperature":       {"table_id": "0000010202", "item_code": "#B02101"},   # average temperature
    # Income and money
    "income_per_taxpayer": {"table_id": "0000010204", "item_code": "#D02206"}, # taxable income per taxpayer
    "household_income":  {"table_id": "0000010212", "item_code": "#L01201"},   # monthly income of worker households
    "savings_rate":      {"table_id": "0000010212", "item_code": "#L02734"},   # household savings rate (%)
    # Jobs
    "unemployment":      {"table_id": "0000010206", "item_code": "#F01301"},   # unemployment rate (%)
    "job_openings":      {"table_id": "0000010206", "item_code": "#F03103"},   # job openings per job seeker
    # Public finance
    "fiscal_strength":   {"table_id": "0000010204", "item_code": "#D0110101"}, # prefecture's own tax strength
    # Education
    "univ_rate":         {"table_id": "0000010205", "item_code": "#E09402"},   # high-school grads going to university (%)
    "univ_capacity":     {"table_id": "0000010205", "item_code": "#E0610202"}, # university places vs local students
    # Culture
    "libraries":         {"table_id": "0000010207", "item_code": "#G01104"},   # libraries per 1 million people
    # Housing
    "homeownership":     {"table_id": "0000010208", "item_code": "#H01301"},   # homes owned (%)
    "vacancy_rate":      {"table_id": "0000010208", "item_code": "#H01405"},   # empty homes (%)
    "rent":              {"table_id": "0000010208", "item_code": "#H04102"},   # private rent (yen per 3.3 m2)
    # Health
    "doctors":           {"table_id": "0000010209", "item_code": "#I0920101"}, # doctors per 100,000 people
    "hospitals":         {"table_id": "0000010209", "item_code": "#I0910103"}, # hospitals per 100,000 people
    # Welfare and childcare
    "childcare_use":     {"table_id": "0000010205", "item_code": "#E0910402"}, # children using nurseries (%)
    "elderly_homes":     {"table_id": "0000010210", "item_code": "#J022011"},  # elderly homes per 100,000 people 65+
    "welfare":           {"table_id": "0000010210", "item_code": "#J01107"},   # people on welfare per 1,000
    # Safety
    "crime":             {"table_id": "0000010211", "item_code": "#K06101"},   # crimes per 1,000 people
    "traffic_accidents": {"table_id": "0000010211", "item_code": "#K04101"},   # traffic accidents per 100,000 people
}

In [ ]:
# Download every statistic in the dictionary (about 1 minute)
dfs = {}

for name, info in tables.items():
    dfs[name] = get_estat(info["table_id"], info["item_code"])
    print(name, "downloaded:", len(dfs[name]), "rows")

In [ ]:
# Combine into one table: one row per area and year, one column per statistic
combined = None

for name, df in dfs.items():
    df = df.rename(columns={"value": name})
    if combined is None:
        combined = df
    else:
        combined = combined.merge(df, on=["area_code", "area", "year"], how="outer")

print(combined.shape)
combined.head()

In [ ]:
# How many values are missing per column (NaN = not collected that year, e.g. census only every 5 years)
combined.isna().sum()

In [ ]:
# Separate the whole of Japan from the 47 prefectures, and turn counts into rates
japan = combined[combined["area"] == "All Japan"].copy()
pref  = combined[combined["area"] != "All Japan"].copy()

for df in [japan, pref]:
    df["aging_rate"]    = df["pop_65plus"] / df["pop"] * 100                     # % aged 65+
    df["natural_rate"]  = (df["births"] - df["deaths"]) / df["pop"] * 1000       # births - deaths per 1,000
    df["net_migration"] = (df["moves_in"] - df["moves_out"]) / df["pop"] * 1000  # moves in - out per 1,000

pref = pref.sort_values(["area_code", "year"])
pref.head()

In [ ]:
# Group prefectures into 8 regions (area_code: 01 = Hokkaido ... 47 = Okinawa, north to south)
def region(code):
    n = int(code[:2])
    if n == 1:   return "Hokkaido"
    if n <= 7:   return "Tohoku"
    if n <= 14:  return "Kanto"
    if n <= 23:  return "Chubu"
    if n <= 30:  return "Kinki"
    if n <= 35:  return "Chugoku"
    if n <= 39:  return "Shikoku"
    return "Kyushu-Okinawa"

pref["region"] = pref["area_code"].apply(region)
pref.groupby("region")["area"].nunique()   # prefectures per region

In [ ]:
# Mark Fukushima after 2011 (nuclear evacuation, not normal population decline)
pref["fukushima_after_2011"] = (pref["area"] == "Fukushima-ken") & (pref["year"] >= 2011)
pref["fukushima_after_2011"].sum()

In [ ]:
# Remove years with no population data (2025 has only migration numbers)
pref = pref.dropna(subset=["pop"])
pref["year"].max()

In [ ]:
# Some surveys are not held in census years (e.g. housing: 1978, 1983, ...).
# Fill each gap with the latest earlier value, at most 4 years old.
features = ["aging_rate", "fertility", "income_per_taxpayer", "unemployment",
            "job_openings", "density", "univ_capacity", "doctors", "vacancy_rate", "snow_days",
            "marriage_rate", "household_income", "fiscal_strength", "rent", "hospitals",
            "childcare_use", "welfare", "crime"]

pref[features] = pref.groupby("area")[features].ffill(limit=4)

In [ ]:
# 5-year table: keep census years and add the population change over the next 5 and 10 years
pref5 = pref[pref["year"] % 5 == 0].copy()

pref5["pop_next5"]   = pref5.groupby("area")["pop"].shift(-1)        # population 5 years later
pref5["pop_next10"]  = pref5.groupby("area")["pop"].shift(-2)        # population 10 years later
pref5["growth_5y"]   = (pref5["pop_next5"]  - pref5["pop"]) / pref5["pop"] * 100    # % change, next 5 years
pref5["growth_10y"]  = (pref5["pop_next10"] - pref5["pop"]) / pref5["pop"] * 100    # % change, next 10 years
pref5["growth_prev"] = pref5.groupby("area")["growth_5y"].shift(1)   # % change in the previous 5 years

pref5[pref5["area"] == "Akita-ken"][["area", "year", "pop", "growth_5y", "growth_10y", "growth_prev"]]

In [ ]:
# Save the prepared tables
pref.to_csv("prefectures_yearly.csv", index=False)
pref5.to_csv("prefectures_5yr.csv", index=False)
japan.to_csv("japan.csv", index=False)

## 2. Feature selection: which statistics matter most?

Three methods each pick their **top 5** statistics. To compare the methods fairly, we train the same simple
model (linear regression) on each method's 5 statistics, **learn from the past (1985–2005) and test on the
most recent periods (2010–2020)**. The method whose statistics predict best wins.

In [ ]:
# Data for feature selection: statistics at the start of each 5-year period -> change over the next 5 years
data = pref5[(pref5["fukushima_after_2011"] == False)].dropna(subset=features + ["growth_5y"])

train = data[data["year"] <= 2005]     # periods starting 1985 ... 2005 (learn)
test  = data[data["year"] >= 2010]     # periods starting 2010, 2015       (test)

print("train:", len(train), "rows, years", sorted(train["year"].unique()))
print("test: ", len(test),  "rows, years", sorted(test["year"].unique()))

In [ ]:
# Method 1: correlation - how strongly each statistic moves with population change (training data only)
corr = train[features + ["growth_5y"]].corr()["growth_5y"].drop("growth_5y")
corr_rank = corr.abs().sort_values(ascending=False)

top_corr = list(corr_rank.index[:5])
print("Top 5 by correlation:", top_corr)

corr_rank.sort_values().plot(kind="barh", title="Method 1: |correlation| with 5-year change")
plt.show()

In [ ]:
# Method 2: random forest - how much the model relies on each statistic to predict the change
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=500, random_state=0)
rf.fit(train[features], train["growth_5y"])
rf_rank = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

top_rf = list(rf_rank.index[:5])
print("Top 5 by random forest:", top_rf)

rf_rank.sort_values().plot(kind="barh", title="Method 2: random forest importance")
plt.show()

In [ ]:
# Method 3: Lasso - a regression that shrinks unimportant statistics to zero
# (statistics are first put on the same scale, so their coefficients can be compared)
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(train[features])

lasso = LassoCV(cv=5, random_state=0).fit(X_scaled, train["growth_5y"])
lasso_rank = pd.Series(abs(lasso.coef_), index=features).sort_values(ascending=False)

top_lasso = list(lasso_rank.index[:5])
print("Top 5 by Lasso:", top_lasso)
print("Statistics Lasso dropped (coefficient = 0):", list(lasso_rank[lasso_rank == 0].index))

lasso_rank.sort_values().plot(kind="barh", title="Method 3: |Lasso coefficient|")
plt.show()

In [ ]:
# Compare the methods: train the same linear regression on each method's top 5, test on 2010-2020
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

candidates = {"Correlation": top_corr, "Random forest": top_rf, "Lasso": top_lasso, "All 18 statistics": features}
results = []

for method, cols in candidates.items():
    model = LinearRegression().fit(train[cols], train["growth_5y"])
    pred = model.predict(test[cols])
    results.append({"method": method,
                    "MAE (percentage points)": mean_absolute_error(test["growth_5y"], pred),
                    "R2": r2_score(test["growth_5y"], pred)})

results = pd.DataFrame(results).set_index("method").round(3)
print(results)

results["MAE (percentage points)"].plot(kind="barh", title="Test error by feature-selection method (lower = better)")
plt.show()

In [ ]:
# Choose the best method (smallest error), not counting "all 18" which is only a reference
best_method = results.drop("All 18 statistics")["MAE (percentage points)"].idxmin()
selected = candidates[best_method]

print("Best method:", best_method)
print("Selected statistics:", selected)

# Which statistics each method picked (1 = picked)
picked = pd.DataFrame({m: [1 if f in cols else 0 for f in features] for m, cols in candidates.items() if m != "All 18 statistics"},
                      index=features)
picked[picked.sum(axis=1) > 0].sort_values(list(picked.columns), ascending=False)

## 2b. Do the leading factors change over time?

The same correlation as Method 1, but calculated **separately for each period**:
about 30, 20 and 10 years ago, and now.

In [ ]:
# Changing factors: correlation with the next 5 years' change, calculated separately for each period
periods = {"1985-1990 (~30 years ago)": 1985, "1995-2000 (~20 years ago)": 1995,
           "2005-2010 (~10 years ago)": 2005, "2015-2020 (now)": 2015}

by_period = {}
for label, year in periods.items():
    d = pref5[(pref5["year"] == year) & (pref5["fukushima_after_2011"] == False)].dropna(subset=["growth_5y"])
    by_period[label] = d[features + ["growth_5y"]].corr()["growth_5y"].drop("growth_5y")

by_period = pd.DataFrame(by_period).round(2)
by_period.sort_values("2015-2020 (now)")

In [ ]:
# Top 3 factors in each period (largest correlation, positive or negative)
for label in by_period.columns:
    top3 = by_period[label].abs().sort_values(ascending=False).index[:3]
    print(label, "->", ", ".join(f"{f} ({by_period.loc[f, label]:+.2f})" for f in top3))

In [ ]:
# Heatmap: blue = more of it goes with growth, red = more of it goes with decline
plt.figure(figsize=(7, 8))
plt.imshow(by_period.values, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="correlation with next 5 years' change")
plt.yticks(range(len(by_period)), by_period.index)
plt.xticks(range(len(by_period.columns)), by_period.columns, rotation=30, ha="right")
for i in range(by_period.shape[0]):
    for j in range(by_period.shape[1]):
        v = by_period.values[i, j]
        if not np.isnan(v):
            plt.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
plt.title("Which factors go with population change, period by period")
plt.tight_layout()
plt.show()

## 3. ANOVA: does population change differ between groups?

Following the 7 steps from class: question → look at the data → check the conditions → F test →
effect size → which groups differ (Tukey) → report.

**Example 1 (one-way ANOVA):** does the 5-year population change (2015→2020) differ between regions?

In [ ]:
# Step 1 - question and design: one value per prefecture (2015 -> 2020), grouped by region
# Hokkaido is a region with only 1 prefecture, so it is joined with Tohoku (common grouping in Japan)
anova_data = pref5[(pref5["year"] == 2015) & (pref5["fukushima_after_2011"] == False)].copy()
anova_data["region"] = anova_data["region"].replace({"Hokkaido": "Hokkaido-Tohoku", "Tohoku": "Hokkaido-Tohoku"})
anova_data = anova_data.dropna(subset=["growth_5y"])

print(len(anova_data), "prefectures")
anova_data["region"].value_counts()

In [ ]:
# Step 2 - look at the data: every prefecture as a dot, group means as black bars; means and SDs
order = anova_data.groupby("region")["growth_5y"].mean().sort_values().index

for i, r in enumerate(order):
    values = anova_data[anova_data["region"] == r]["growth_5y"]
    plt.scatter([i] * len(values), values, alpha=0.7)
    plt.hlines(values.mean(), i - 0.3, i + 0.3, color="black", linewidth=2)

plt.axhline(anova_data["growth_5y"].mean(), color="gray", linestyle="--", label="grand mean")
plt.xticks(range(len(order)), order, rotation=30)
plt.ylabel("Population change 2015-2020 (%)")
plt.title("Population change by region (black bar = mean)")
plt.legend()
plt.show()

anova_data.groupby("region")["growth_5y"].agg(["count", "mean", "std"]).round(2).loc[order]

In [ ]:
# Step 3 - check the conditions: equal spread (largest SD / smallest SD < 2) and bell-shaped noise (Q-Q plot)
import statsmodels.api as sm

sds = anova_data.groupby("region")["growth_5y"].std()
print("SD ratio (largest / smallest):", round(sds.max() / sds.min(), 2), "(rule of thumb: below 2)")

residuals = anova_data["growth_5y"] - anova_data.groupby("region")["growth_5y"].transform("mean")
sm.qqplot(residuals, line="s")
plt.title("Q-Q plot of residuals (points near the line = bell-shaped)")
plt.show()

In [ ]:
# Step 4 - the F test: ANOVA table (between regions vs. within regions)
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy import stats

model = smf.ols("growth_5y ~ C(region)", data=anova_data).fit()
anova_table = anova_lm(model)
print(anova_table.round(4))

# Back-up test that uses ranks instead of values (if the checks in step 3 look doubtful)
groups = [g["growth_5y"].values for _, g in anova_data.groupby("region")]
kruskal_p = stats.kruskal(*groups).pvalue
print("\nKruskal-Wallis (rank test): p =", round(kruskal_p, 6))

In [ ]:
# Step 5 - effect size: eta squared = share of all variation that lies between the regions
ss_between = anova_table.loc["C(region)", "sum_sq"]
ss_total = anova_table["sum_sq"].sum()
eta_sq = ss_between / ss_total
print("eta squared =", round(eta_sq, 2), "->", round(eta_sq * 100), "% of the variation goes with the region")

In [ ]:
# Step 6 - which regions differ? Tukey's HSD (keeps the overall false-alarm rate at 5%)
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(anova_data["growth_5y"], anova_data["region"])
print(tukey)

tukey.plot_simultaneous(xlabel="Population change 2015-2020 (%)")
plt.title("Tukey: region means with 95% intervals (no overlap = significantly different)")
plt.show()

In [ ]:
# Step 7 - report the result in one paragraph
df_b = int(anova_table.loc["C(region)", "df"])
df_w = int(anova_table.loc["Residual", "df"])
F = anova_table.loc["C(region)", "F"]
p = anova_table.loc["C(region)", "PR(>F)"]
n_sig = int(tukey.reject.sum())

print(f"One-way ANOVA, population change 2015-2020 by region, {len(anova_data)} prefectures: "
      f"F({df_b}, {df_w}) = {F:.2f}, p = {p:.4f}; eta squared = {eta_sq:.2f}. "
      f"Tukey's HSD: {n_sig} of {len(tukey.reject)} region pairs differ significantly. "
      f"Checks: SD ratio {sds.max() / sds.min():.1f} (rule of thumb: below 2), so the rank test was also run: "
      f"Kruskal-Wallis p = {kruskal_p:.4f}, the same conclusion. Q-Q plot: see above.")

**Example 3 from class (same units measured repeatedly):** the same 46 prefectures are measured in every
5-year period. Did population change differ between periods? Each prefecture's own level is removed first.

In [ ]:
# Repeated measures: every prefecture is one line over the periods; the thick line is the mean
rm_data = pref5[(pref5["area"] != "Fukushima-ken")].dropna(subset=["growth_5y"])[["area", "year", "growth_5y"]]

for area, g in rm_data.groupby("area"):
    plt.plot(g["year"], g["growth_5y"], color="steelblue", alpha=0.25)
mean_line = rm_data.groupby("year")["growth_5y"].mean()
plt.plot(mean_line.index, mean_line.values, color="darkorange", linewidth=3, marker="o", label="mean of 46 prefectures")
plt.axhline(0, color="gray", linewidth=1)
plt.xlabel("Start of 5-year period")
plt.ylabel("Population change over 5 years (%)")
plt.title("Each line is one prefecture measured over time")
plt.legend()
plt.show()

In [ ]:
# Repeated-measures ANOVA: does the period matter, after removing each prefecture's own level?
# (statsmodels does not give the corrected p-value from class; with an F this large the conclusion would not change)
from statsmodels.stats.anova import AnovaRM

rm = AnovaRM(rm_data, depvar="growth_5y", subject="area", within=["year"]).fit()
print(rm)

In [ ]:
# Which periods differ? Paired t tests between neighbouring periods, with Bonferroni's rule (p x number of tests)
wide = rm_data.pivot(index="area", columns="year", values="growth_5y")
years = list(wide.columns)
rows = []

for a, b in zip(years[:-1], years[1:]):
    t, p = stats.ttest_rel(wide[b], wide[a])
    rows.append({"pair": f"{b} vs {a}", "mean change (pp)": (wide[b] - wide[a]).mean(),
                 "t": t, "p x tests (Bonferroni)": min(p * (len(years) - 1), 1)})

pd.DataFrame(rows).round(4)

## 4. Prediction: population 5 and 10 years ahead

Inputs: the statistics chosen in Part 2, plus the population change in the previous 5 years (the recent trend).
Each model is tested on years it has not seen: **5 years** = forecast made in 2015 for 2020,
**10 years** = forecast made in 2010 for 2020.

In [ ]:
# Inputs and the rows we can use (Fukushima after 2011 left out)
inputs = selected + ["growth_prev"]
model_data = pref5[pref5["fukushima_after_2011"] == False].dropna(subset=inputs)
print("Inputs:", inputs)

In [ ]:
# A function that trains and tests three models: naive, linear regression, random forest
def compare_models(target, test_year, years_ahead):
    rows = model_data.dropna(subset=[target])
    train = rows[rows["year"] + years_ahead <= test_year]      # only outcomes known at the time of the forecast
    test  = rows[rows["year"] == test_year]

    predictions = {
        # naive: the last 5 years' change simply repeats
        "Naive (trend repeats)": (1 + test["growth_prev"] / 100) ** (years_ahead / 5) * 100 - 100,
        "Linear regression": LinearRegression().fit(train[inputs], train[target]).predict(test[inputs]),
        "Random forest": RandomForestRegressor(n_estimators=500, random_state=0)
                             .fit(train[inputs], train[target]).predict(test[inputs]),
    }
    scores = pd.DataFrame({name: {"MAE (pp)": mean_absolute_error(test[target], p),
                                  "R2": r2_score(test[target], p)} for name, p in predictions.items()}).T
    return scores.round(3), test, predictions

In [ ]:
# 5 years ahead: learn from periods up to 2010 -> 2015, test the forecast 2015 -> 2020
scores5, test5, pred5 = compare_models("growth_5y", test_year=2015, years_ahead=5)
print(scores5)

best5 = scores5["MAE (pp)"].idxmin()
plt.scatter(pred5[best5], test5["growth_5y"])
lims = [test5["growth_5y"].min() - 1, test5["growth_5y"].max() + 1]
plt.plot(lims, lims, color="gray")
plt.xlabel(f"Predicted change 2015-2020 (%) - {best5}")
plt.ylabel("Actual change 2015-2020 (%)")
plt.title("5-year forecast vs. what happened (on the line = perfect)")
plt.show()

In [ ]:
# 10 years ahead: learn from periods up to 2000 -> 2010, test the forecast 2010 -> 2020
scores10, test10, pred10 = compare_models("growth_10y", test_year=2010, years_ahead=10)
print(scores10)

best10 = scores10["MAE (pp)"].idxmin()
plt.scatter(pred10[best10], test10["growth_10y"])
lims = [test10["growth_10y"].min() - 1, test10["growth_10y"].max() + 1]
plt.plot(lims, lims, color="gray")
plt.xlabel(f"Predicted change 2010-2020 (%) - {best10}")
plt.ylabel("Actual change 2010-2020 (%)")
plt.title("10-year forecast vs. what happened")
plt.show()

In [ ]:
# Compare the errors of all models side by side
pd.DataFrame({"5 years": scores5["MAE (pp)"], "10 years": scores10["MAE (pp)"]}).plot(
    kind="bar", title="Forecast error by model (lower = better)", ylabel="MAE (percentage points)", rot=0)
plt.show()
print("Best 5-year model:", best5, "| Best 10-year model:", best10)

In [ ]:
# Forecast the future from 2020: retrain the best models on all known periods
def fit_best(name, target):
    rows = model_data.dropna(subset=[target])
    if name == "Linear regression":
        return LinearRegression().fit(rows[inputs], rows[target])
    if name == "Random forest":
        return RandomForestRegressor(n_estimators=500, random_state=0).fit(rows[inputs], rows[target])
    return None   # naive needs no training

now = model_data[model_data["year"] == 2020].copy()
for name, target, years_ahead, col in [(best5, "growth_5y", 5, "growth_2025"), (best10, "growth_10y", 10, "growth_2030")]:
    model = fit_best(name, target)
    if model is None:
        now[col] = (1 + now["growth_prev"] / 100) ** (years_ahead / 5) * 100 - 100
    else:
        now[col] = model.predict(now[inputs])

now["pop_2025"] = now["pop"] * (1 + now["growth_2025"] / 100)
now["pop_2030"] = now["pop"] * (1 + now["growth_2030"] / 100)

forecast = now[["area", "region", "pop", "growth_2025", "pop_2025", "growth_2030", "pop_2030"]].sort_values("growth_2030")
forecast.round(1)

In [ ]:
# Chart: forecast change 2020 -> 2030 for every prefecture
f = forecast.sort_values("growth_2030")
plt.figure(figsize=(7, 11))
plt.barh(f["area"], f["growth_2030"], color=["tab:red" if g < 0 else "tab:blue" for g in f["growth_2030"]])
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Forecast population change 2020-2030 (%)")
plt.title("Forecast by prefecture (red = shrinking)")
plt.tight_layout()
plt.show()

In [ ]:
# Totals for the 46 prefectures, and a check against the real 2024 population
print(f"2020: {forecast['pop'].sum() / 1e6:.2f} million")
print(f"2025 forecast: {forecast['pop_2025'].sum() / 1e6:.2f} million")
print(f"2030 forecast: {forecast['pop_2030'].sum() / 1e6:.2f} million")

actual_2024 = pref[pref["year"] == 2024].set_index("area")["pop"]
check = forecast.set_index("area")[["pop", "growth_2025"]].copy()
check["actual_change_2020_2024"] = (actual_2024 / check["pop"] - 1) * 100
check["forecast_change_2020_2024"] = ((1 + check["growth_2025"] / 100) ** 0.8 - 1) * 100   # 4 of the 5 years

print("\nCheck against the real 2024 population:")
print("  average error:", round((check["forecast_change_2020_2024"] - check["actual_change_2020_2024"]).abs().mean(), 2), "percentage points")
print("  correlation:  ", round(check["forecast_change_2020_2024"].corr(check["actual_change_2020_2024"]), 2))